# Parcel3D — ANN → SNN (production notebook)

Self-contained pipeline: CNN → standard SNN → calibrated SNN → STBP fine-tune. Run **top to bottom**.

**Assumptions:** Kaggle paths below; GPU optional (CPU fallbacks). Hardcoded paths are intentional.

| Block | Content |
|-------|---------|
| Setup | Paths, deps, config |
| Data | Cached Parcel3D loaders |
| CNN | Load or train (multi-seed) |
| Metrics | CNN test + energy |
| SNN | Conversion, sweeps, STBP |
| Export | Tables, figures, inventory |


---
## 1 — Paths and directories

In [ ]:
import gc
import json
import os
import pickle
import random
import sys
import time
import warnings
from pathlib import Path

# Narrow warnings (avoid blanket suppression)
warnings.filterwarnings(
    "ignore",
    message=".*pkg_resources is deprecated.*",
    category=UserWarning,
)

# ── Kaggle dataset paths (fixed as requested) ───────────────────────────────
DATA_ROOT = Path("/kaggle/input/datasets/ayesha19765/parcel3d/parcel3d")
SAVED_CNN_ROOT = Path("/kaggle/input/datasets/adityakvermaa/parcel3d-cnn-weights/outputs")
OUTPUT_DIR = Path("/kaggle/working/outputs")

for d in ("cnn", "snn", "figures", "cache"):
    (OUTPUT_DIR / d).mkdir(parents=True, exist_ok=True)

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Parcel3D dataset missing: {DATA_ROOT}")

print("DATA_ROOT :", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
SAVED_WEIGHTS_AVAILABLE = SAVED_CNN_ROOT.exists()
print("Saved CNN weights:", "FOUND" if SAVED_WEIGHTS_AVAILABLE else "NOT FOUND — will train")


---
## 2 — Dependencies

In [ ]:
import subprocess
import sys

pkgs = ["spikingjelly==0.0.0.0.14", "thop", "seaborn"]
for pkg in pkgs:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        print(f"WARNING: pip install failed for {pkg}: {(r.stderr or '')[:300]}")

import numpy as np
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  VRAM={p.total_memory / 1024**3:.1f} GiB")


---
## 3 — Config, SpikingJelly, utilities

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as T
from PIL import Image

try:
    from spikingjelly.clock_driven import functional as sj_functional
    from spikingjelly.clock_driven import ann2snn as sj_ann2snn
    from spikingjelly.clock_driven.neuron import IFNode, LIFNode
    SJ_API = "clock_driven"
except ImportError:
    from spikingjelly.activation_based import functional as sj_functional
    from spikingjelly.activation_based import ann2snn as sj_ann2snn
    from spikingjelly.activation_based.neuron import IFNode, LIFNode
    SJ_API = "activation_based"

print("SpikingJelly API:", SJ_API)

CFG = {
    "IMG_SIZE": 224,
    "BATCH_SIZE": 32,
    "NUM_WORKERS": 0,
    "NUM_CLASSES": 2,
    "CLASS_NAMES": ["normal", "damaged"],
    "SEEDS": [42, 123, 456],
    "STAGE1_EPOCHS": 3,
    "STAGE2_EPOCHS": 10,
    "PATIENCE": 3,
    "DROPOUT": 0.25,
    "LABEL_SMOOTH": 0.05,
    "LR_STAGE1": 1e-3,
    "LR_STAGE2": 1e-4,
    "GRAD_CLIP": 1.0,
    "T_VALUES": [32, 64, 128, 256, 512],
    "SNN_PERCENTILE": 99.9,
    "CALIB_BATCHES": 16,
    "TARGET_RATE": 0.10,
    "MIN_SCALE": 0.70,
    "MAX_SCALE": 1.30,
    "STBP_T": 128,
    "STBP_EPOCHS": 5,
    "STBP_LR": 5e-5,
    "STBP_BATCH": 16,
    "E_MAC_PJ": 4.6,
    "E_AC_PJ": 0.9,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def vram_report(label=""):
    if not torch.cuda.is_available():
        print("  [VRAM] N/A (CPU mode)")
        return
    free_b, total_b = torch.cuda.mem_get_info()
    used_gib = (total_b - free_b) / (1024**3)
    total_gib = total_b / (1024**3)
    free_gib = free_b / (1024**3)
    tag = f" {label}" if label else ""
    print(f"  [VRAM{tag}] {used_gib:.2f}/{total_gib:.2f} GiB used ({free_gib:.2f} GiB free)")


def load_torch_blob(path, map_location):
    # Load torch.save; supports PyTorch with/without weights_only kwarg.
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        pass
    except Exception:
        pass
    return torch.load(path, map_location=map_location, weights_only=False)


def extract_state_dict(obj):
    if isinstance(obj, dict):
        if "model_state_dict" in obj:
            return obj["model_state_dict"]
        if "state_dict" in obj:
            return obj["state_dict"]
        return obj
    return obj


def resolve_fan_out(if_name: str, cost_model: dict):
    # Map IF node name to fan-out; deterministic fallbacks.
    if not cost_model:
        return 512.0
    if if_name in cost_model:
        return float(cost_model[if_name])
    for k, v in cost_model.items():
        if k in if_name or if_name in k:
            return float(v)
    return float(np.mean(list(cost_model.values())))


print("Config (scalars):", {k: v for k, v in CFG.items() if not isinstance(v, list)})


---
## 4 — Dataset and loaders

In [ ]:
class Parcel3DDataset(Dataset):
    LABEL_MAP = {"normal": 0, "damaged": 1}

    def __init__(self, root: Path, split: str, transform=None, cache_dir: Path = None):
        self.transform = transform
        self.samples = []
        split_dir = root / split
        if not split_dir.is_dir():
            raise FileNotFoundError(f"Split directory missing: {split_dir}")

        cache_key = str(split_dir).replace("/", "_").replace("\\", "_")
        cache_path = (cache_dir / f"{cache_key}.json") if cache_dir else None

        if cache_path and cache_path.exists():
            with open(cache_path, encoding="utf-8") as f:
                raw = json.load(f)
            self.samples = [(Path(p), lbl) for p, lbl in raw]
            print(f"  [{split}] loaded {len(self.samples)} paths from cache")
        else:
            print(f"  [{split}] scanning …")
            t0 = time.time()
            for class_name, label in self.LABEL_MAP.items():
                class_dir = split_dir / class_name
                if not class_dir.exists():
                    print(f"    WARNING: missing class dir: {class_dir}")
                    continue
                found = sorted(class_dir.rglob("rgb.png"))
                if not found:
                    found = (
                        sorted(class_dir.glob("*.png"))
                        + sorted(class_dir.glob("*.jpg"))
                        + sorted(class_dir.glob("*.JPEG"))
                    )
                for p in found:
                    self.samples.append((p, label))
            print(f"  [{split}] {len(self.samples)} images in {time.time() - t0:.1f}s")
            if cache_path:
                cache_path.parent.mkdir(parents=True, exist_ok=True)
                with open(cache_path, "w", encoding="utf-8") as f:
                    json.dump([(str(p), lbl) for p, lbl in self.samples], f)
                print(f"  [{split}] index cached → {cache_path}")

        if len(self.samples) == 0:
            raise RuntimeError(f"No images found under {split_dir}")

        labels = [lbl for _, lbl in self.samples]
        for lbl, name in enumerate(("normal", "damaged")):
            cnt = labels.count(lbl)
            pct = 100.0 * cnt / len(labels) if labels else 0.0
            print(f"    {name}: {cnt} ({pct:.1f}%)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"WARNING: failed to open {path}: {e}")
            img = Image.new("RGB", (CFG["IMG_SIZE"], CFG["IMG_SIZE"]), color=0)
        if self.transform:
            img = self.transform(img)
        return img, label


def build_transforms(img_size: int):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    train_tf = T.Compose(
        [
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            T.ToTensor(),
            T.Normalize(mean, std),
        ]
    )
    eval_tf = T.Compose(
        [
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean, std),
        ]
    )
    return train_tf, eval_tf


def get_dataloaders(data_root, img_size, batch_size, num_workers, cache_dir=None):
    train_tf, eval_tf = build_transforms(img_size)
    loaders, sizes = {}, {}
    for split, tf, shuffle in (
        ("train", train_tf, True),
        ("validation", eval_tf, False),
        ("test", eval_tf, False),
    ):
        ds = Parcel3DDataset(data_root, split, transform=tf, cache_dir=cache_dir)
        pin = torch.cuda.is_available()
        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin,
            drop_last=False,
            persistent_workers=(num_workers > 0),
        )
        sizes[split] = len(ds)
    return loaders, sizes


print("Building data loaders …")
loaders, sizes = get_dataloaders(
    DATA_ROOT,
    img_size=CFG["IMG_SIZE"],
    batch_size=CFG["BATCH_SIZE"],
    num_workers=CFG["NUM_WORKERS"],
    cache_dir=OUTPUT_DIR / "cache",
)
print("Split sizes:", sizes)
for k, n in sizes.items():
    if n == 0:
        raise RuntimeError(f"Split '{k}' has zero samples")

imgs, lbls = next(iter(loaders["train"]))
if imgs.shape[0] < 2:
    print("WARNING: batch smaller than expected (dataset may be tiny).")
print(f"Train batch: {imgs.shape}  labels: {lbls.shape}  dtype: {imgs.dtype}")


---
## 5 — Model: ParcelVGG

In [ ]:
class ParcelVGG(nn.Module):
    def __init__(self, num_classes: int = 2, pretrained: bool = True, dropout: float = 0.25):
        super().__init__()
        try:
            backbone = torchvision.models.vgg11_bn(weights="IMAGENET1K_V1")
        except TypeError:
            backbone = torchvision.models.vgg11_bn(pretrained=pretrained)

        features = []
        for layer in backbone.features:
            if isinstance(layer, nn.MaxPool2d):
                features.append(nn.AvgPool2d(kernel_size=2, stride=2))
            else:
                features.append(layer)
        self.features = nn.Sequential(*features)

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )
        self._freeze_all()

    def _freeze_all(self):
        for p in self.parameters():
            p.requires_grad = False

    def prepare_stage1(self):
        self._freeze_all()
        for p in self.head.parameters():
            p.requires_grad = True

    def prepare_stage2(self):
        for p in self.head.parameters():
            p.requires_grad = True
        for i, layer in enumerate(self.features):
            if i >= 15:
                for p in layer.parameters():
                    p.requires_grad = True

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True

    def param_groups(self, lr_head: float, lr_backbone: float):
        backbone_params = [
            p
            for i, layer in enumerate(self.features)
            if i >= 15
            for p in layer.parameters()
            if p.requires_grad
        ]
        head_params = [p for p in self.head.parameters() if p.requires_grad]
        return [
            {"params": backbone_params, "lr": lr_backbone},
            {"params": head_params, "lr": lr_head},
        ]

    def verify_ann2snn_compat(self) -> bool:
        relu_count, bad_layers = 0, []
        for name, m in self.named_modules():
            if isinstance(m, nn.ReLU):
                relu_count += 1
            if isinstance(m, (nn.MaxPool2d, nn.GELU, nn.SiLU, nn.Sigmoid)):
                bad_layers.append((name, type(m).__name__))
        ok = len(bad_layers) == 0
        print(f"ann2snn compat: {'PASS' if ok else 'FAIL'}  relu_count={relu_count}")
        if bad_layers:
            print("  Incompatible layers:", bad_layers)
        return ok

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.head(x)
        return x


with torch.no_grad():
    _m = ParcelVGG(num_classes=CFG["NUM_CLASSES"]).to(DEVICE)
    _m.eval()
    assert _m.verify_ann2snn_compat()
    _dummy = torch.randn(2, 3, CFG["IMG_SIZE"], CFG["IMG_SIZE"], device=DEVICE)
    _out = _m(_dummy)
    assert _out.shape == (2, CFG["NUM_CLASSES"])
    del _m, _dummy, _out
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("ParcelVGG OK")


---
## 6 — CNN training / checkpoint loading

In [ ]:
from sklearn.metrics import f1_score


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=device.type == "cuda"):
            out = model(imgs)
            loss = criterion(out, lbls)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        with autocast(enabled=device.type == "cuda"):
            out = model(imgs)
            loss = criterion(out, lbls)
        total_loss += loss.item() * imgs.size(0)
        all_preds += out.argmax(1).cpu().tolist()
        all_labels += lbls.cpu().tolist()
    n = len(all_labels)
    if n == 0:
        return 0.0, 0.0
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / n, f1


def train_one_seed(seed: int, loaders: dict, device: torch.device, cfg: dict) -> dict:
    set_seed(seed)
    ckpt_path = OUTPUT_DIR / "cnn" / f"best_seed_{seed}.pth"

    model = ParcelVGG(num_classes=cfg["NUM_CLASSES"]).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["LABEL_SMOOTH"])
    scaler = GradScaler(enabled=(device.type == "cuda"))

    best_f1, patience_ctr, log = 0.0, 0, []

    model.prepare_stage1()
    optimizer = torch.optim.AdamW(model.param_groups(cfg["LR_STAGE1"], 0.0), weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["STAGE1_EPOCHS"])

    for ep in range(1, cfg["STAGE1_EPOCHS"] + 1):
        tr_loss, tr_acc = train_one_epoch(model, loaders["train"], optimizer, criterion, scaler, device)
        va_loss, va_f1 = validate(model, loaders["validation"], criterion, device)
        scheduler.step()
        log.append(
            {
                "seed": seed,
                "epoch": ep,
                "stage": 1,
                "tr_loss": tr_loss,
                "tr_acc": tr_acc,
                "va_loss": va_loss,
                "va_f1": va_f1,
            }
        )
        print(
            f"  S1 ep{ep:2d} | tr_loss={tr_loss:.4f} acc={tr_acc:.4f} | val_loss={va_loss:.4f} f1={va_f1:.4f}"
        )
        if va_f1 > best_f1:
            best_f1 = va_f1
            torch.save(model.state_dict(), ckpt_path)

    model.prepare_stage2()
    optimizer = torch.optim.AdamW(
        model.param_groups(cfg["LR_STAGE2"], cfg["LR_STAGE2"] * 0.1),
        weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["STAGE2_EPOCHS"])
    patience_ctr = 0

    for ep in range(1, cfg["STAGE2_EPOCHS"] + 1):
        tr_loss, tr_acc = train_one_epoch(model, loaders["train"], optimizer, criterion, scaler, device)
        va_loss, va_f1 = validate(model, loaders["validation"], criterion, device)
        scheduler.step()
        log.append(
            {
                "seed": seed,
                "epoch": cfg["STAGE1_EPOCHS"] + ep,
                "stage": 2,
                "tr_loss": tr_loss,
                "tr_acc": tr_acc,
                "va_loss": va_loss,
                "va_f1": va_f1,
            }
        )
        print(
            f"  S2 ep{ep:2d} | tr_loss={tr_loss:.4f} acc={tr_acc:.4f} | val_loss={va_loss:.4f} f1={va_f1:.4f}"
        )
        if va_f1 > best_f1:
            best_f1 = va_f1
            patience_ctr = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f"         ↑ new best: {best_f1:.4f} → saved")
        else:
            patience_ctr += 1
            if patience_ctr >= cfg["PATIENCE"]:
                print(f"  Early stop at ep {ep} (patience={cfg['PATIENCE']})")
                break

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"seed": seed, "best_val_f1": best_f1, "ckpt_path": str(ckpt_path), "log": log}


In [ ]:
import csv

seed_results = {}
all_logs = []
WEIGHTS_LOADED = False
cnn = None

if SAVED_WEIGHTS_AVAILABLE:
    cnn_dir = SAVED_CNN_ROOT / "cnn"
    if cnn_dir.is_dir():
        candidates = sorted(cnn_dir.glob("*.pth"))
        preferred = [p for p in candidates if p.stem == "best_cnn_overall"]
        ordered = preferred + [p for p in candidates if p not in preferred]
        print(f"Found {len(candidates)} .pth file(s): {[p.name for p in candidates]}")
        for pth in ordered:
            try:
                cnn = ParcelVGG(num_classes=CFG["NUM_CLASSES"]).to(DEVICE)
                cnn.unfreeze_all()
                obj = load_torch_blob(pth, map_location=DEVICE)
                sd = extract_state_dict(obj)
                cnn.load_state_dict(sd, strict=True)
                print(f"✓ Loaded CNN weights from {pth.name}")
                WEIGHTS_LOADED = True
                for s in CFG["SEEDS"]:
                    seed_results[s] = {"best_val_f1": None, "ckpt_path": str(pth)}
                break
            except Exception as e:
                print(f"  ✗ {pth.name}: {e}")

if not WEIGHTS_LOADED:
    local_ckpts = {s: OUTPUT_DIR / "cnn" / f"best_seed_{s}.pth" for s in CFG["SEEDS"]}
    if all(p.exists() for p in local_ckpts.values()):
        best_f1, best_seed, best_path = -1.0, None, None
        for s, p in local_ckpts.items():
            try:
                obj = load_torch_blob(p, map_location="cpu")
                f1 = float(obj.get("best_val_f1", -1.0)) if isinstance(obj, dict) else -1.0
                seed_results[s] = {"best_val_f1": f1, "ckpt_path": str(p)}
                if f1 > best_f1:
                    best_f1, best_seed, best_path = f1, s, p
            except Exception:
                continue
        if best_path is not None:
            cnn = ParcelVGG(num_classes=CFG["NUM_CLASSES"]).to(DEVICE)
            cnn.unfreeze_all()
            obj = load_torch_blob(best_path, map_location=DEVICE)
            sd = extract_state_dict(obj)
            cnn.load_state_dict(sd, strict=True)
            WEIGHTS_LOADED = True
            print(f"✓ Resumed from local checkpoint seed={best_seed} f1={best_f1:.4f}")

if not WEIGHTS_LOADED:
    print("\nNo pre-trained weights found — training from scratch.")
    for seed in CFG["SEEDS"]:
        print("\n" + "─" * 60 + f"\nSeed {seed}\n" + "─" * 60)
        res = train_one_seed(seed, loaders, DEVICE, CFG)
        seed_results[seed] = res
        all_logs.extend(res["log"])

    log_path = OUTPUT_DIR / "cnn" / "training_log.csv"
    if all_logs:
        keys = list(all_logs[0].keys())
        with open(log_path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(all_logs)
        print(f"Training log → {log_path}")

    valid_results = {s: r for s, r in seed_results.items() if r.get("best_val_f1") is not None}
    if not valid_results:
        raise RuntimeError("Training finished but no valid seed results.")
    best_seed = max(valid_results, key=lambda s: valid_results[s]["best_val_f1"])
    best_f1 = valid_results[best_seed]["best_val_f1"]
    best_ckpt = Path(valid_results[best_seed]["ckpt_path"])
    print(f"\nBest seed: {best_seed}  val_f1={best_f1:.4f}")

    import shutil
    overall_ckpt = OUTPUT_DIR / "cnn" / "best_cnn_overall.pth"
    shutil.copy(best_ckpt, overall_ckpt)

    cnn = ParcelVGG(num_classes=CFG["NUM_CLASSES"]).to(DEVICE)
    cnn.unfreeze_all()
    sd = load_torch_blob(best_ckpt, map_location=DEVICE)
    sd = extract_state_dict(sd)
    cnn.load_state_dict(sd, strict=True)
    WEIGHTS_LOADED = True

if cnn is None:
    raise RuntimeError("CNN model is not initialized.")

with open(OUTPUT_DIR / "cnn" / "seed_results.json", "w", encoding="utf-8") as f:
    json.dump(
        {str(k): {kk: vv for kk, vv in v.items() if kk != "log"} for k, v in seed_results.items()},
        f,
        indent=2,
    )

cnn.eval()
print("\nCNN ready on", DEVICE)
vram_report("CNN ready")


---
## 7 — CNN evaluation and energy

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
import thop


@torch.no_grad()
def evaluate_cnn(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    lat_times = []
    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        t0 = time.perf_counter()
        out = model(imgs)
        cuda_sync()
        lat_times.append((time.perf_counter() - t0) / max(imgs.size(0), 1) * 1000.0)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        preds = out.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_probs.extend(probs.tolist())
        all_labels.extend(lbls.numpy().tolist())

    if len(all_labels) == 0:
        raise RuntimeError("evaluate_cnn: empty loader")
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = 0.5
    cm = confusion_matrix(all_labels, all_preds)
    lat = float(np.mean(lat_times[1:])) if len(lat_times) > 1 else float(np.mean(lat_times))
    return {
        "accuracy": float(acc),
        "f1_macro": float(f1),
        "auc_roc": float(auc),
        "confusion_matrix": cm.tolist(),
        "latency_ms": lat,
        "all_preds": all_preds,
        "all_labels": all_labels,
        "all_probs": all_probs,
    }


def compute_cnn_energy(model, device, img_size):
    model.eval()
    dummy = torch.randn(1, 3, img_size, img_size, device=device)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        macs, params = thop.profile(model, inputs=(dummy,), verbose=False)
    energy_pj = float(macs) * CFG["E_MAC_PJ"]
    return {"macs": int(macs), "params": int(params), "energy_uJ": float(energy_pj / 1e6)}


def build_activation_cost_model(model, device, img_size):
    model.eval()
    cost_model = {}
    all_layers = list(model.named_modules())
    for i, (name, m) in enumerate(all_layers):
        if not isinstance(m, nn.ReLU):
            continue
        fan_out = None
        for j in range(i + 1, min(i + 8, len(all_layers))):
            _, next_m = all_layers[j]
            if isinstance(next_m, nn.Conv2d):
                kh, kw = next_m.kernel_size
                fan_out = int(next_m.out_channels * kh * kw)
                break
            if isinstance(next_m, nn.Linear):
                fan_out = int(next_m.out_features)
                break
        if fan_out is None:
            fan_out = 512
        cost_model[name] = fan_out
    return cost_model


print("Evaluating CNN on test set …")
t0 = time.time()
cnn_metrics = evaluate_cnn(cnn, loaders["test"], DEVICE)
print(f"  Elapsed: {time.time() - t0:.1f}s")
print(f"  Accuracy : {cnn_metrics['accuracy'] * 100:.2f}%")
print(f"  F1 macro : {cnn_metrics['f1_macro'] * 100:.2f}%")
print(f"  AUC-ROC  : {cnn_metrics['auc_roc'] * 100:.2f}%")
print(f"  Latency  : {cnn_metrics['latency_ms']:.2f} ms/img")

cnn_energy = compute_cnn_energy(cnn, DEVICE, CFG["IMG_SIZE"])
print(f"\n  MACs     : {cnn_energy['macs'] / 1e9:.3f} G")
print(f"  Energy   : {cnn_energy['energy_uJ']:.2f} µJ/img")

cost_model = build_activation_cost_model(cnn, DEVICE, CFG["IMG_SIZE"])
print(f"\n  ReLU layers tracked : {len(cost_model)}")
if cost_model:
    print(f"  Avg fan-out         : {np.mean(list(cost_model.values())):.0f}")

with open(OUTPUT_DIR / "cnn" / "cnn_test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(
        {k: v for k, v in cnn_metrics.items() if k not in ("all_preds", "all_labels", "all_probs")},
        f,
        indent=2,
    )


---
## 8 — SNN conversion, evaluation, energy

In [ ]:
def convert_standard(ann_model, calib_loader, device, percentile=99.9):
    model_copy = copy.deepcopy(ann_model).to(device)
    model_copy.eval()
    mode_str = f"{percentile}%"
    try:
        converter = sj_ann2snn.Converter(
            dataloader=calib_loader,
            device=device,
            mode=mode_str,
            momentum=0.1,
            fuse_flag=True,
        )
    except TypeError:
        converter = sj_ann2snn.Converter(dataloader=calib_loader, device=device, mode=mode_str)
    print(f"  Running {mode_str} calibration on {len(calib_loader)} batches …")
    snn = converter(model_copy)
    snn.to(device)
    snn.eval()
    return snn


@torch.no_grad()
def evaluate_snn_at_T(snn, loader, device, T):
    snn.eval()
    all_preds, all_labels, all_probs = [], [], []
    spike_totals, spike_counts = {}, {}
    lat_times = []
    hooks = []

    def _make_spike_hook(name):
        def _hook(module, inp, out):
            spike_totals[name] = spike_totals.get(name, 0.0) + out.float().mean().item()
            spike_counts[name] = spike_counts.get(name, 0) + 1

        return _hook

    for name, m in snn.named_modules():
        if isinstance(m, IFNode):
            hooks.append(m.register_forward_hook(_make_spike_hook(name)))

    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        sj_functional.reset_net(snn)
        t0 = time.perf_counter()
        acc = torch.zeros(imgs.size(0), CFG["NUM_CLASSES"], device=device, dtype=torch.float32)
        for _ in range(T):
            acc = acc + snn(imgs)
        cuda_sync()
        lat_times.append((time.perf_counter() - t0) / max(imgs.size(0), 1) * 1000.0)
        logits = acc / float(T)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_probs.extend(probs.tolist())
        all_labels.extend(lbls.numpy().tolist())

    for h in hooks:
        h.remove()

    if len(all_labels) == 0:
        raise RuntimeError("evaluate_snn_at_T: empty loader")

    acc_val = accuracy_score(all_labels, all_preds)
    f1_val = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    try:
        auc_val = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc_val = 0.5

    per_layer_rates = {n: spike_totals[n] / max(spike_counts[n], 1) for n in spike_totals}
    mean_rate = float(np.mean(list(per_layer_rates.values()))) if per_layer_rates else 0.0
    lat = float(np.mean(lat_times[1:])) if len(lat_times) > 1 else float(np.mean(lat_times))

    return {
        "T": T,
        "accuracy": float(acc_val),
        "f1_macro": float(f1_val),
        "auc_roc": float(auc_val),
        "mean_spike_rate": mean_rate,
        "per_layer_rates": per_layer_rates,
        "latency_ms": lat,
        "all_preds": all_preds,
        "all_labels": all_labels,
        "all_probs": all_probs,
    }


def compute_snn_energy(snn_results: dict, cost_model: dict):
    energies = {}
    for T, res in snn_results.items():
        per_layer = res["per_layer_rates"]
        total_synops = 0.0
        for if_name, rate in per_layer.items():
            fan_out = float(resolve_fan_out(if_name, cost_model))
            total_synops += rate * fan_out
        energy_uj = total_synops * CFG["E_AC_PJ"] / 1e6
        energies[T] = {
            "energy_uJ": float(energy_uj),
            "total_synops": float(total_synops),
            "spike_rate": float(res["mean_spike_rate"]),
        }
    return energies


def sweep_T(snn, loader, device, T_values, label=""):
    results = {}
    for T in T_values:
        print(f"  {label} T={T:4d} …", end=" ", flush=True)
        t0 = time.time()
        r = evaluate_snn_at_T(snn, loader, device, T)
        results[T] = r
        print(
            f"acc={r['accuracy'] * 100:.2f}%  f1={r['f1_macro'] * 100:.2f}%  "
            f"auc={r['auc_roc'] * 100:.2f}%  spike={r['mean_spike_rate'] * 100:.2f}%  "
            f"({time.time() - t0:.0f}s)"
        )
    return results


---
## 9 — Standard SNN + T sweep

In [ ]:
print("\nConverting ANN → SNN (standard) …")
t0 = time.time()
cnn.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
vram_report("before std conversion")

snn_std = convert_standard(cnn.to(DEVICE), loaders["validation"], DEVICE, percentile=CFG["SNN_PERCENTILE"])
cnn.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
vram_report("after std conversion")
print(f"  Conversion done in {time.time() - t0:.1f}s")

print("\nT sweep — standard SNN:")
results_std = sweep_T(snn_std, loaders["test"], DEVICE, CFG["T_VALUES"], label="Std")
energies_std = compute_snn_energy(results_std, cost_model)

cnn_uJ = cnn_energy["energy_uJ"]
print(f"\n{'T':>6}  {'Acc':>8}  {'F1':>8}  {'AUC':>8}  {'E(µJ)':>10}  {'Ratio':>8}  {'Rate':>8}")
print("─" * 70)
for T in CFG["T_VALUES"]:
    m = results_std[T]
    e = energies_std[T]
    ratio = cnn_uJ / e["energy_uJ"] if e["energy_uJ"] > 0 else 0.0
    print(
        f"  {T:>4}  {m['accuracy'] * 100:>7.2f}%  {m['f1_macro'] * 100:>7.2f}%  "
        f"{m['auc_roc'] * 100:>7.2f}%  {e['energy_uJ']:>10.4f}  {ratio:>7.1f}×  {m['mean_spike_rate'] * 100:>7.2f}%"
    )

with open(OUTPUT_DIR / "results_standard.pkl", "wb") as f:
    pickle.dump({"results": results_std, "energies": energies_std}, f)

del snn_std
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
vram_report("after freeing snn_std")


---
## 10 — Calibrated SNN + T sweep

In [ ]:
def measure_ifnode_firing_rates(snn, loader, device, T_cal, max_batches=16):
    snn.eval()
    totals, counts = {}, {}
    hooks = []

    def _make_hook(name):
        def _hook(module, inp, out):
            totals[name] = totals.get(name, 0.0) + out.float().mean().item()
            counts[name] = counts.get(name, 0) + 1

        return _hook

    for name, m in snn.named_modules():
        if isinstance(m, IFNode):
            hooks.append(m.register_forward_hook(_make_hook(name)))

    with torch.no_grad():
        for batch_idx, (imgs, _) in enumerate(loader):
            if batch_idx >= max_batches:
                break
            imgs = imgs.to(device, non_blocking=True)
            sj_functional.reset_net(snn)
            for _ in range(T_cal):
                snn(imgs)

    for h in hooks:
        h.remove()

    return {n: totals[n] / max(counts[n], 1) for n in totals}


def apply_firing_rate_calibration(snn, observed_rates, target_rate, min_scale, max_scale):
    n_adjusted = 0
    print(f"  {'Layer':<50} {'Observed':>10} {'Scale':>8} {'New thresh':>12}")
    print("  " + "─" * 82)
    for name, m in snn.named_modules():
        if not isinstance(m, IFNode):
            continue
        if name not in observed_rates:
            continue
        obs = observed_rates[name]
        if obs < 1e-8:
            scale = min_scale
        else:
            scale = float(np.clip(obs / target_rate, min_scale, max_scale))
        old_thr = float(m.v_threshold)
        new_thr = old_thr * scale
        m.v_threshold = new_thr
        n_adjusted += 1
        print(f"  {name:<50} {obs * 100:>9.2f}%  {scale:>7.3f}×  {new_thr:>12.4f}")
    print(f"\n  Adjusted {n_adjusted} IFNode thresholds")
    return snn


print("Converting ANN → SNN (calibrated pipeline) …")
print(f"  target_rate   = {CFG['TARGET_RATE'] * 100:.0f}%")
print(f"  scale bounds  = [{CFG['MIN_SCALE']}, {CFG['MAX_SCALE']}]")
print(f"  calib_batches = {CFG['CALIB_BATCHES']}")

vram_report("before cal conversion")
cnn.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

t0 = time.time()
snn_cal = convert_standard(cnn.to(DEVICE), loaders["validation"], DEVICE, percentile=CFG["SNN_PERCENTILE"])
cnn.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"  Base conversion done in {time.time() - t0:.1f}s")
vram_report("after base conversion")

T_cal = 128
print(f"\n  Measuring per-layer firing rates (T={T_cal}, {CFG['CALIB_BATCHES']} batches) …")
observed = measure_ifnode_firing_rates(
    snn_cal, loaders["validation"], DEVICE, T_cal=T_cal, max_batches=CFG["CALIB_BATCHES"]
)
if observed:
    print(f"  Overall mean rate: {np.mean(list(observed.values())) * 100:.2f}%")
else:
    print("  WARNING: no IFNode activity captured — check hooks / model.")

print("\n  Adjusting thresholds …")
snn_cal = apply_firing_rate_calibration(
    snn_cal,
    observed,
    target_rate=CFG["TARGET_RATE"],
    min_scale=CFG["MIN_SCALE"],
    max_scale=CFG["MAX_SCALE"],
)

cal_report = {
    "target_rate": CFG["TARGET_RATE"],
    "T_cal": T_cal,
    "calib_batches": CFG["CALIB_BATCHES"],
    "min_scale": CFG["MIN_SCALE"],
    "max_scale": CFG["MAX_SCALE"],
    "observed_rates": {k: float(v) for k, v in observed.items()},
}
with open(OUTPUT_DIR / "snn" / "calibration_report.json", "w", encoding="utf-8") as f:
    json.dump(cal_report, f, indent=2)

print("\nT sweep — calibrated SNN:")
results_cal = sweep_T(snn_cal, loaders["test"], DEVICE, CFG["T_VALUES"], label="Cal")
energies_cal = compute_snn_energy(results_cal, cost_model)

print(f"\n{'T':>6}  {'Acc':>8}  {'F1':>8}  {'AUC':>8}  {'E(µJ)':>10}  {'Ratio':>8}  {'Rate':>8}")
print("─" * 70)
for T in CFG["T_VALUES"]:
    m = results_cal[T]
    e = energies_cal[T]
    ratio = cnn_uJ / e["energy_uJ"] if e["energy_uJ"] > 0 else 0.0
    print(
        f"  {T:>4}  {m['accuracy'] * 100:>7.2f}%  {m['f1_macro'] * 100:>7.2f}%  "
        f"{m['auc_roc'] * 100:>7.2f}%  {e['energy_uJ']:>10.4f}  {ratio:>7.1f}×  {m['mean_spike_rate'] * 100:>7.2f}%"
    )

with open(OUTPUT_DIR / "results_calibrated.pkl", "wb") as f:
    pickle.dump({"results": results_cal, "energies": energies_cal, "cal_report": cal_report}, f)

cal_state_dict = copy.deepcopy(snn_cal.state_dict())
torch.save(cal_state_dict, OUTPUT_DIR / "snn" / "snn_calibrated.pth")
print(f"\n  Saved calibrated SNN → {OUTPUT_DIR / 'snn' / 'snn_calibrated.pth'}")

del snn_cal
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
vram_report("after freeing snn_cal")


---
## 11 — STBP fine-tuning + T sweep

In [ ]:
def stbp_train_epoch(snn, loader, optimizer, criterion, scaler, device, T_train):
    snn.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=device.type == "cuda"):
            sj_functional.reset_net(snn)
            acc = torch.zeros(
                imgs.size(0), CFG["NUM_CLASSES"], device=device, dtype=torch.float32
            )
            for _ in range(T_train):
                acc = acc + snn(imgs)
            logits = acc / float(T_train)
            loss = criterion(logits, lbls)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(snn.parameters(), CFG["GRAD_CLIP"])
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.detach().argmax(1) == lbls).sum().item()
        total += imgs.size(0)
        sj_functional.reset_net(snn)
    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def stbp_validate(snn, loader, device, T_train):
    snn.eval()
    all_preds, all_labels = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        sj_functional.reset_net(snn)
        acc = torch.zeros(
            imgs.size(0), CFG["NUM_CLASSES"], device=device, dtype=torch.float32
        )
        for _ in range(T_train):
            acc = acc + snn(imgs)
        preds = (acc / float(T_train)).argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(lbls.numpy().tolist())
    if len(all_labels) == 0:
        return 0.0
    return f1_score(all_labels, all_preds, average="macro", zero_division=0)


print("Building STBP SNN from calibrated init …")
cnn.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

snn_ft = convert_standard(cnn.to(DEVICE), loaders["validation"], DEVICE, percentile=CFG["SNN_PERCENTILE"])
cnn.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

snn_ft.load_state_dict(cal_state_dict)
snn_ft.to(DEVICE)
print("  Calibrated weights loaded.")
vram_report("before STBP training")

stbp_loader = DataLoader(
    loaders["train"].dataset,
    batch_size=CFG["STBP_BATCH"],
    shuffle=True,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

stbp_optimizer = torch.optim.Adam(snn_ft.parameters(), lr=CFG["STBP_LR"], weight_decay=1e-5)
stbp_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    stbp_optimizer, T_max=CFG["STBP_EPOCHS"], eta_min=CFG["STBP_LR"] * 0.1
)
stbp_criterion = nn.CrossEntropyLoss(label_smoothing=CFG["LABEL_SMOOTH"])
stbp_scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

T_TRAIN = CFG["STBP_T"]
print(f"\nSTBP: T={T_TRAIN}, epochs={CFG['STBP_EPOCHS']}, batch={CFG['STBP_BATCH']}, LR={CFG['STBP_LR']}")

best_val_f1_stbp = -1.0
stbp_log = []
best_path = OUTPUT_DIR / "snn" / "snn_stbp_best.pth"

for ep in range(1, CFG["STBP_EPOCHS"] + 1):
    t0 = time.time()
    tr_loss, tr_acc = stbp_train_epoch(
        snn_ft, stbp_loader, stbp_optimizer, stbp_criterion, stbp_scaler, DEVICE, T_TRAIN
    )
    val_f1 = stbp_validate(snn_ft, loaders["validation"], DEVICE, T_TRAIN)
    stbp_scheduler.step()
    elapsed = time.time() - t0
    stbp_log.append({"epoch": ep, "tr_loss": tr_loss, "tr_acc": tr_acc, "val_f1": val_f1})
    print(
        f"  ep{ep:2d}/{CFG['STBP_EPOCHS']}  tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  "
        f"val_f1={val_f1:.4f}  ({elapsed:.0f}s)"
    )
    improved = val_f1 > best_val_f1_stbp
    if improved:
        best_val_f1_stbp = val_f1
    if improved or ep == 1:
        torch.save(
            {"model_state_dict": snn_ft.state_dict(), "epoch": ep, "val_f1": val_f1, "T_train": T_TRAIN},
            best_path,
        )
        print(f"         ↑ checkpoint saved (val_f1={val_f1:.4f})")

best_stbp = load_torch_blob(best_path, map_location=DEVICE)
snn_ft.load_state_dict(extract_state_dict(best_stbp))
if isinstance(best_stbp, dict) and "val_f1" in best_stbp:
    print(f"\nLoaded STBP checkpoint (val_f1={best_stbp['val_f1']:.4f})")
else:
    print("\nLoaded STBP checkpoint")

print("\nT sweep — STBP SNN:")
results_ft = sweep_T(snn_ft, loaders["test"], DEVICE, CFG["T_VALUES"], label="STBP")
energies_ft = compute_snn_energy(results_ft, cost_model)

print(f"\n{'T':>6}  {'Acc':>8}  {'F1':>8}  {'AUC':>8}  {'E(µJ)':>10}  {'Ratio':>8}  {'Rate':>8}")
print("─" * 70)
for T in CFG["T_VALUES"]:
    m = results_ft[T]
    e = energies_ft[T]
    ratio = cnn_uJ / e["energy_uJ"] if e["energy_uJ"] > 0 else 0.0
    print(
        f"  {T:>4}  {m['accuracy'] * 100:>7.2f}%  {m['f1_macro'] * 100:>7.2f}%  "
        f"{m['auc_roc'] * 100:>7.2f}%  {e['energy_uJ']:>10.4f}  {ratio:>7.1f}×  {m['mean_spike_rate'] * 100:>7.2f}%"
    )

with open(OUTPUT_DIR / "results_stbp.pkl", "wb") as f:
    pickle.dump({"results": results_ft, "energies": energies_ft, "stbp_log": stbp_log}, f)


---
## 12 — Summary tables and JSON

In [ ]:
import csv


class SafeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, Path):
            return str(obj)
        return super().default(obj)


def safe_json_dump(path, payload):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, cls=SafeEncoder)


def row_dict(variant, T, metrics, energy_uJ, cnn_uj):
    ratio = cnn_uj / energy_uJ if energy_uJ > 0 else 0.0
    return {
        "variant": variant,
        "T": T,
        "accuracy_pct": round(metrics["accuracy"] * 100, 2),
        "f1_pct": round(metrics["f1_macro"] * 100, 2),
        "auc_pct": round(metrics["auc_roc"] * 100, 2),
        "energy_uJ": round(energy_uJ, 4),
        "ratio_vs_cnn": round(ratio, 1),
        "spike_rate_pct": round(metrics["mean_spike_rate"] * 100, 2),
        "latency_ms": round(metrics["latency_ms"], 2),
    }


rows = []
rows.append(
    {
        "variant": "CNN",
        "T": "-",
        "accuracy_pct": round(cnn_metrics["accuracy"] * 100, 2),
        "f1_pct": round(cnn_metrics["f1_macro"] * 100, 2),
        "auc_pct": round(cnn_metrics["auc_roc"] * 100, 2),
        "energy_uJ": round(cnn_energy["energy_uJ"], 4),
        "ratio_vs_cnn": 1.0,
        "spike_rate_pct": 0.0,
        "latency_ms": round(cnn_metrics["latency_ms"], 2),
    }
)

for T in CFG["T_VALUES"]:
    rows.append(row_dict("Standard", T, results_std[T], energies_std[T]["energy_uJ"], cnn_uJ))
    rows.append(row_dict("Calibrated", T, results_cal[T], energies_cal[T]["energy_uJ"], cnn_uJ))
    rows.append(row_dict("STBP", T, results_ft[T], energies_ft[T]["energy_uJ"], cnn_uJ))

print("=" * 110)
print("FULL RESULTS: CNN vs Standard vs Calibrated vs STBP")
print("=" * 110)
print(
    f"  {'Variant':<14} {'T':>5}  {'Acc%':>7}  {'F1%':>7}  {'AUC%':>7}  "
    f"{'E(µJ)':>10}  {'Ratio':>8}  {'Rate%':>7}  {'Lat(ms)':>9}"
)
print("─" * 110)
last_var = None
for r in rows:
    if r["variant"] != last_var and last_var is not None:
        print("─" * 110)
    last_var = r["variant"]
    print(
        f"  {r['variant']:<14} {str(r['T']):>5}  {r['accuracy_pct']:>6.2f}%  {r['f1_pct']:>6.2f}%  "
        f"{r['auc_pct']:>6.2f}%  {r['energy_uJ']:>10.4f}  {r['ratio_vs_cnn']:>7.1f}×  "
        f"{r['spike_rate_pct']:>6.2f}%  {r['latency_ms']:>9.2f}"
    )
print("=" * 110)

csv_path = OUTPUT_DIR / "full_results_table.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
print(f"\nResults table → {csv_path}")

safe_json_dump(
    OUTPUT_DIR / "cnn_metrics.json",
    {
        "accuracy": cnn_metrics["accuracy"],
        "f1_macro": cnn_metrics["f1_macro"],
        "auc_roc": cnn_metrics["auc_roc"],
        "energy_uJ": cnn_energy["energy_uJ"],
        "macs": cnn_energy["macs"],
        "latency_ms": cnn_metrics["latency_ms"],
    },
)

for name, res, eng in (
    ("results_standard_summary.json", results_std, energies_std),
    ("results_calibrated_summary.json", results_cal, energies_cal),
    ("results_stbp_summary.json", results_ft, energies_ft),
):
    safe_json_dump(
        OUTPUT_DIR / name,
        {
            T: {
                "accuracy": res[T]["accuracy"],
                "f1_macro": res[T]["f1_macro"],
                "auc_roc": res[T]["auc_roc"],
                "energy_uJ": eng[T]["energy_uJ"],
                "spike_rate": res[T]["mean_spike_rate"],
                "latency_ms": res[T]["latency_ms"],
            }
            for T in CFG["T_VALUES"]
        },
    )
print("JSON summaries saved.")


---
## 13 — Figures

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 200,
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
    }
)

PALETTE = {
    "CNN": "#E45756",
    "Standard": "#4C78A8",
    "Calibrated": "#F58518",
    "STBP": "#54A24B",
}
MARKERS = {"Standard": "o", "Calibrated": "s", "STBP": "^"}

Ts = CFG["T_VALUES"]
cnn_acc = cnn_metrics["accuracy"] * 100
cnn_f1 = cnn_metrics["f1_macro"] * 100
cnn_auc = cnn_metrics["auc_roc"] * 100
cnn_E = cnn_energy["energy_uJ"]


def _save(fig, stem):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{stem}.{ext}", bbox_inches="tight")
    plt.close(fig)
    print(f"  ✓  {stem}.png / .pdf")


# Fig 1: Accuracy / F1 vs T — CNN shown as reference lines only (same x-axis as SNN curves)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Parcel3D: Accuracy & Macro-F1 vs Timestep T", fontweight="bold")

for ax, (key, ylabel, cnn_y, cnn_lbl) in zip(
    axes,
    [
        ("accuracy", "Test Accuracy (%)", cnn_acc, f"CNN acc={cnn_acc:.1f}%"),
        ("f1_macro", "Test Macro-F1 (%)", cnn_f1, f"CNN macro-F1={cnn_f1:.1f}%"),
    ],
):
    ax.axhline(cnn_y, color=PALETTE["CNN"], lw=1.8, ls=":", label=cnn_lbl)
    for variant, res in (
        ("Standard", results_std),
        ("Calibrated", results_cal),
        ("STBP", results_ft),
    ):
        vals = [res[T][key] * 100 for T in Ts]
        ax.plot(
            Ts,
            vals,
            color=PALETTE[variant],
            marker=MARKERS[variant],
            lw=2,
            ms=8,
            label=variant,
        )
        for T, v in zip(Ts, vals):
            ax.annotate(
                f"{v:.1f}",
                (T, v),
                textcoords="offset points",
                xytext=(4, 4),
                fontsize=7.5,
                color=PALETTE[variant],
            )
    ax.set_xlabel("Timestep T")
    ax.set_ylabel(ylabel)
    ax.set_xscale("log", base=2)
    ax.set_xticks(Ts)
    ax.set_xticklabels([str(t) for t in Ts])
    ax.set_ylim(30, 103)
    ax.legend()

plt.tight_layout()
_save(fig, "fig1_accuracy_f1_vs_T")

# Fig 2: AUC vs T
fig, ax = plt.subplots(figsize=(7, 5))
ax.set_title("Parcel3D: AUC-ROC vs Timestep T", fontweight="bold")
ax.axhline(cnn_auc, color=PALETTE["CNN"], lw=1.8, ls=":", label=f"CNN AUC={cnn_auc:.2f}%")
for variant, res in (("Standard", results_std), ("Calibrated", results_cal), ("STBP", results_ft)):
    aucs = [res[T]["auc_roc"] * 100 for T in Ts]
    ax.plot(Ts, aucs, color=PALETTE[variant], marker=MARKERS[variant], lw=2, ms=8, label=variant)
    for T, a in zip(Ts, aucs):
        ax.annotate(
            f"{a:.1f}", (T, a), textcoords="offset points", xytext=(4, 4), fontsize=7.5, color=PALETTE[variant]
        )
ax.set_xlabel("Timestep T")
ax.set_ylabel("AUC-ROC (%)")
ax.set_xscale("log", base=2)
ax.set_xticks(Ts)
ax.set_xticklabels([str(t) for t in Ts])
ax.set_ylim(40, 103)
ax.legend()
plt.tight_layout()
_save(fig, "fig2_auc_vs_T")

# Fig 3: Energy–accuracy (Pareto-style)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Parcel3D: Energy–Accuracy Trade-off", fontweight="bold")
for ax, (key, ylabel) in zip(
    axes,
    [("accuracy", "Test Accuracy (%)"), ("f1_macro", "Test Macro-F1 (%)")],
):
    ax.scatter(
        [cnn_E],
        [cnn_metrics[key] * 100],
        color=PALETTE["CNN"],
        marker="*",
        s=280,
        zorder=6,
        label=f"CNN ({cnn_metrics[key]*100:.1f}%, {cnn_E:.3f} µJ)",
    )
    for variant, res, engs in (
        ("Standard", results_std, energies_std),
        ("Calibrated", results_cal, energies_cal),
        ("STBP", results_ft, energies_ft),
    ):
        Es = [engs[T]["energy_uJ"] for T in Ts]
        vals = [res[T][key] * 100 for T in Ts]
        ax.plot(
            Es,
            vals,
            color=PALETTE[variant],
            marker=MARKERS[variant],
            ls="--",
            lw=1.8,
            ms=7,
            label=variant,
        )
        for T, E, v in zip(Ts, Es, vals):
            ax.annotate(
                f"T={T}",
                (E, v),
                textcoords="offset points",
                xytext=(4, -12),
                fontsize=7,
                color=PALETTE[variant],
            )
    ax.set_xlabel("Energy (µJ / image)")
    ax.set_ylabel(ylabel)
    ax.set_xscale("log")
    ax.set_ylim(30, 103)
    ax.legend(loc="lower right")

plt.tight_layout()
_save(fig, "fig3_pareto_front")

# Fig 4: Energy reduction ratio
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_title("Parcel3D: Energy Reduction Ratio (CNN / SNN) vs T", fontweight="bold")
bar_w = 0.25
x = np.arange(len(Ts))
offsets = [-bar_w, 0, bar_w]
for offset, (variant, engs) in zip(
    offsets,
    (("Standard", energies_std), ("Calibrated", energies_cal), ("STBP", energies_ft)),
):
    ratios = [cnn_E / engs[T]["energy_uJ"] if engs[T]["energy_uJ"] > 0 else 0 for T in Ts]
    bars = ax.bar(
        x + offset,
        ratios,
        width=bar_w,
        color=PALETTE[variant],
        alpha=0.85,
        label=variant,
        edgecolor="white",
    )
    for bar, r in zip(bars, ratios):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{r:.0f}×",
            ha="center",
            fontsize=7.5,
            fontweight="bold",
        )
ax.set_xticks(x)
ax.set_xticklabels([f"T={t}" for t in Ts])
ax.set_ylabel("Energy reduction vs CNN (×)")
ax.legend()
plt.tight_layout()
_save(fig, "fig4_energy_reduction")

# Fig 5: Spike rate + latency
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Parcel3D: Spike Rate & Latency vs T", fontweight="bold")
for variant, res in (("Standard", results_std), ("Calibrated", results_cal), ("STBP", results_ft)):
    rates = [res[T]["mean_spike_rate"] * 100 for T in Ts]
    lats = [res[T]["latency_ms"] for T in Ts]
    ax1.plot(Ts, rates, color=PALETTE[variant], marker=MARKERS[variant], lw=2, ms=8, label=variant)
    ax2.plot(Ts, lats, color=PALETTE[variant], marker=MARKERS[variant], lw=2, ms=8, label=variant)
ax1.axhline(
    CFG["TARGET_RATE"] * 100,
    color="grey",
    ls=":",
    lw=1.5,
    label=f"Target ({CFG['TARGET_RATE']*100:.0f}%)",
)
ax2.axhline(
    cnn_metrics["latency_ms"],
    color=PALETTE["CNN"],
    ls=":",
    lw=1.8,
    label=f"CNN ({cnn_metrics['latency_ms']:.1f} ms)",
)
for ax, ylabel in ((ax1, "Mean spike rate (%)"), (ax2, "Latency (ms / image)")):
    ax.set_xlabel("Timestep T")
    ax.set_ylabel(ylabel)
    ax.set_xscale("log", base=2)
    ax.set_xticks(Ts)
    ax.set_xticklabels([str(t) for t in Ts])
    ax.legend()
plt.tight_layout()
_save(fig, "fig5_spike_rate_latency")


def best_T_by_f1(results):
    return max(results, key=lambda t: results[t]["f1_macro"])


best_T_std = best_T_by_f1(results_std)
best_T_cal = best_T_by_f1(results_cal)
best_T_ft = best_T_by_f1(results_ft)

cases = [
    ("CNN", cnn_metrics, None),
    (f"Standard (T={best_T_std})", results_std[best_T_std], best_T_std),
    (f"Calibrated (T={best_T_cal})", results_cal[best_T_cal], best_T_cal),
    (f"STBP (T={best_T_ft})", results_ft[best_T_ft], best_T_ft),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle("Parcel3D: Confusion matrices — CNN vs best SNN per variant", fontweight="bold")
for ax, (title, res, _T_val) in zip(axes, cases):
    labels = res.get("all_labels", [])
    preds = res.get("all_preds", [])
    if not labels or not preds:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        continue
    cm = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=CFG["CLASS_NAMES"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    acc = sum(1 for l, p in zip(labels, preds) if l == p) / len(labels)
    ax.set_title(f"{title}\nacc={acc*100:.1f}%", fontsize=10)
plt.tight_layout()
_save(fig, "fig6_confusion_matrices")

variants = ["CNN", "Standard", "Calibrated", "STBP"]
best_res = {
    "CNN": {
        "accuracy": cnn_metrics["accuracy"],
        "f1_macro": cnn_metrics["f1_macro"],
        "auc_roc": cnn_metrics["auc_roc"],
    },
    "Standard": results_std[best_T_std],
    "Calibrated": results_cal[best_T_cal],
    "STBP": results_ft[best_T_ft],
}
colors = [PALETTE[v] for v in variants]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Parcel3D: Best F1 timestep per SNN variant vs CNN", fontweight="bold")
for ax, (metric, mlabel) in zip(
    axes,
    [("accuracy", "Test accuracy (%)"), ("f1_macro", "Test macro-F1 (%)"), ("auc_roc", "AUC-ROC (%)")],
):
    vals = [best_res[v][metric] * 100 for v in variants]
    bars = ax.bar(variants, vals, color=colors, alpha=0.87, edgecolor="white", width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{v:.1f}%",
            ha="center",
            fontsize=10,
            fontweight="bold",
        )
    ax.set_title(mlabel)
    ax.set_ylim(max(0, min(vals) - 10), 103)
plt.tight_layout()
_save(fig, "fig7_summary_best_per_variant")

print(f"\nAll figures saved to {FIG_DIR}")


---
## 14 — Inventory

In [ ]:
print("=" * 65)
print("OUTPUT INVENTORY")
print("=" * 65)
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        sz_kb = p.stat().st_size / 1024
        print(f"  {str(p.relative_to(OUTPUT_DIR)):<55}  {sz_kb:>8.1f} kB")

print()
json_files = sorted(OUTPUT_DIR.rglob("*.json"))
ok, fail = 0, 0
for jpath in json_files:
    try:
        with open(jpath, encoding="utf-8") as f:
            json.load(f)
        print(f"  ✓  {jpath.relative_to(OUTPUT_DIR)}")
        ok += 1
    except Exception as e:
        print(f"  ✗  {jpath.relative_to(OUTPUT_DIR)}  ERROR: {e}")
        fail += 1
print(f"\nJSON: {ok} OK, {fail} failed")
vram_report("final")
print("\nNotebook run complete.")
